# My RAG App – Step 1: Calling Gemini with a Prompt

This notebook walks step by step through sending an **input** to Google's **Gemini** model using a **prompt template** and getting the **output** back.

**Steps:**
1. Install the Gemini SDK
2. Import libraries
3. Configure the Gemini API key
4. Create the Gemini client and choose a model
5. Define a prompt template
6. Provide an input
7. Build the final prompt
8. Call Gemini and read the output
9. Wrap it all in a reusable function

## Step 1: Install the Gemini SDK

`google-genai` is Google's official Python SDK for the Gemini API.
Run this once; skip it if the package is already installed.

In [ ]:
%pip install -q google-genai

## Step 2: Import libraries

- `os` reads the API key from an environment variable.
- `getpass` lets you type the key without it being shown or saved in the notebook.
- `genai` is the Gemini client, and `types` holds the configuration objects.

In [ ]:
import os
from getpass import getpass

from google import genai
from google.genai import types

## Step 3: Configure the Gemini API key

Get a free key from **Google AI Studio**: https://aistudio.google.com/app/apikey

The safest way to provide it is an environment variable named `GEMINI_API_KEY`:
```powershell
setx GEMINI_API_KEY "your-key-here"   # then restart VS Code
```
If the variable isn't set, the cell below asks you to paste the key.

> ⚠️ Never hard-code your API key in the notebook. It is easy to share it by accident.

In [ ]:
api_key = os.environ.get("GEMINI_API_KEY") or getpass("Enter your Gemini API key: ")
print("API key loaded:", "yes" if api_key else "no")

## Step 4: Create the Gemini client and choose a model

The **client** handles all communication with the Gemini API.
`gemini-2.5-flash` is fast and inexpensive, which suits learning. You can switch to `gemini-2.5-pro` for harder tasks.

In [ ]:
client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-2.5-flash"
print("Client ready. Using model:", MODEL_NAME)

## Step 5: Define a prompt template

A **prompt template** is reusable text with a placeholder (`{user_input}`) that is filled in with each input.
It tells the model **who it is**, **what to do**, and **how to format the answer**.

In a full RAG app, this template will later also include a `{context}` placeholder for retrieved documents.

In [ ]:
PROMPT_TEMPLATE = """You are a helpful and knowledgeable assistant.

Answer the user's question clearly and concisely.
- Use simple language.
- Use bullet points where they help.
- If you are not sure, say so instead of guessing.

Question: {user_input}

Answer:"""

## Step 6: Provide an input

This is the question or text you want the model to work on. Change it to anything you like.

In [ ]:
user_input = "What is Retrieval-Augmented Generation (RAG) and why is it useful?"

## Step 7: Build the final prompt

Fill the placeholder in the template with the input. Printing the result shows exactly what is sent to Gemini.

In [ ]:
final_prompt = PROMPT_TEMPLATE.format(user_input=user_input)
print(final_prompt)

## Step 8: Call Gemini and read the output

`generate_content` sends the prompt to the model.
- `temperature` controls creativity: `0` is focused and repeatable, `1` is more creative.
- `max_output_tokens` limits the length of the answer.

`response.text` holds the generated answer.

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=final_prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=1024,
    ),
)

print(response.text)

### (Optional) Show the answer as formatted Markdown

Gemini often replies in Markdown, so rendering it makes the output easier to read.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(response.text))

## Step 9: Wrap it all in a reusable function

Combine Steps 5–8 into one function so you can call `ask_gemini("...")` with any input.

In [ ]:
def ask_gemini(user_input: str, temperature: float = 0.3) -> str:
    """Fill the prompt template with the input, send it to Gemini, and return the answer text."""
    prompt = PROMPT_TEMPLATE.format(user_input=user_input)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature, max_output_tokens=1024),
    )
    return response.text

In [ ]:
answer = ask_gemini("Explain the difference between a vector database and a normal database.")
display(Markdown(answer))

## Next steps toward RAG

You now have the **Generation** part of RAG. To add **Retrieval**:
1. Load your documents (PDFs, text files, and so on).
2. Split them into chunks.
3. Create embeddings for the chunks with Gemini's embedding model (`text-embedding-004` / `gemini-embedding-001`).
4. Store the embeddings in a vector store (for example FAISS or ChromaDB).
5. For each question, retrieve the most relevant chunks and add them to the prompt as `{context}`.